# **Semana 10: Big Data - Análisis en Databricks con NYC Taxi (NB2 - Práctico/Ejercicios)**

## **Propósito de la Sesión**
Aplicar técnicas de Big Data con PySpark SQL sobre un dataset real de gran tamaño (NYC Taxi) disponible en los datasets de muestra de Databricks. Realizaremos agregaciones, consultas SQL y visualizaciones directamente en el notebook.

### **Objetivos de Aprendizaje**
Al finalizar este notebook, el estudiante será capaz de:
1. **Cargar** un dataset grande desde los datasets de muestra de Databricks.
2. **Explorar** la estructura y propiedades del dataset.
3. **Realizar** agregaciones complejas utilizando PySpark SQL.
4. **Crear** visualizaciones rápidas dentro del notebook de Databricks.
5. **Analizar** patrones de viajes en taxi en Nueva York.

## **Configuración Inicial**

### **Nota importante:**
Este notebook está diseñado para ejecutarse en **Databricks Community Edition**. Los datasets de muestra están preinstalados en la plataforma. Si estás usando otra distribución de Spark, los pasos para cargar los datos pueden variar.

### **Instrucciones en Databricks:**
1. Asegúrate de tener un cluster en ejecución (ver sesión NB1).
2. Crea un nuevo notebook y adjúntalo a tu cluster.
3. Copia y pega las celdas de código en tu notebook de Databricks.
4. Ejecuta las celdas secuencialmente.

In [ ]:
# Verificar que estamos en Databricks
try:
    display(dbutils.fs.ls("/databricks-datasets"))
    print("✅ Entorno Databricks detectado. Los datasets de muestra están disponibles.")
except:
    print("⚠️  No se detecta el entorno Databricks. Algunas rutas pueden no estar disponibles.")
    print("   Si estás en otro entorno, necesitarás descargar el dataset manualmente.")

---
## **Parte 1: Carga del Dataset NYC Taxi**

Databricks incluye datasets de muestra en la ruta `/databricks-datasets`. Utilizaremos el dataset de viajes de taxi de Nueva York (NYC Taxi), que contiene millones de registros de viajes en taxi amarillo.

In [ ]:
# Ruta del dataset en Databricks
taxi_path = "/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2020-*.csv"

print(f"Cargando dataset desde: {taxi_path}")

# Leer los archivos CSV (múltiples archivos por mes)
df_taxi = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(taxi_path)

# Mostrar información básica
print(f"\nDataset cargado: {df_taxi.count():,} filas")
print(f"Número de columnas: {len(df_taxi.columns)}")

# Mostrar esquema
print("\nEsquema del dataset:")
df_taxi.printSchema()

### **Exploración inicial del dataset**

Veamos las primeras filas para entender la estructura de los datos.

In [ ]:
# Mostrar primeras 10 filas
display(df_taxi.limit(10))

# Estadísticas descriptivas de algunas columnas numéricas
display(df_taxi.select("trip_distance", "fare_amount", "tip_amount", "total_amount").summary())

---
## **Parte 2: Agregaciones con PySpark SQL**

Ahora realizaremos consultas analíticas utilizando PySpark SQL. Primero, crearemos una vista temporal para poder usar SQL directamente.

In [ ]:
# Crear vista temporal para consultas SQL
df_taxi.createOrReplaceTempView("taxi_viajes")

print("Vista 'taxi_viajes' creada. Ya puedes ejecutar consultas SQL.")

### **Consulta 1: Distribución de viajes por día de la semana**

Extraeremos el día de la semana de la fecha de recogida (`tpep_pickup_datetime`) y contaremos los viajes.

In [ ]:
query_dia_semana = """
SELECT 
    DAYOFWEEK(tpep_pickup_datetime) AS dia_semana_num,
    CASE DAYOFWEEK(tpep_pickup_datetime)
        WHEN 1 THEN 'Domingo'
        WHEN 2 THEN 'Lunes'
        WHEN 3 THEN 'Martes'
        WHEN 4 THEN 'Miércoles'
        WHEN 5 THEN 'Jueves'
        WHEN 6 THEN 'Viernes'
        WHEN 7 THEN 'Sábado'
    END AS dia_semana,
    COUNT(*) AS num_viajes,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio,
    ROUND(AVG(total_amount), 2) AS monto_promedio
FROM taxi_viajes
WHERE tpep_pickup_datetime IS NOT NULL
GROUP BY dia_semana_num
ORDER BY dia_semana_num
"""

df_dia_semana = spark.sql(query_dia_semana)
display(df_dia_semana)

### **Consulta 2: Viajes por hora del día**

Analicemos cómo se distribuyen los viajes a lo largo del día.

In [ ]:
query_hora = """
SELECT 
    HOUR(tpep_pickup_datetime) AS hora,
    COUNT(*) AS num_viajes,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio,
    ROUND(AVG(total_amount), 2) AS monto_promedio,
    ROUND(AVG(tip_amount), 2) AS propina_promedio
FROM taxi_viajes
WHERE tpep_pickup_datetime IS NOT NULL
GROUP BY hora
ORDER BY hora
"""

df_hora = spark.sql(query_hora)
display(df_hora)

### **Consulta 3: Top 10 rutas más frecuentes**

Identificaremos las combinaciones de zonas de recogida y destino más comunes.

In [ ]:
query_rutas = """
SELECT 
    PULocationID AS zona_recogida,
    DOLocationID AS zona_destino,
    COUNT(*) AS num_viajes,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio,
    ROUND(AVG(total_amount), 2) AS monto_promedio
FROM taxi_viajes
WHERE PULocationID IS NOT NULL AND DOLocationID IS NOT NULL
GROUP BY PULocationID, DOLocationID
ORDER BY num_viajes DESC
LIMIT 10
"""

df_rutas = spark.sql(query_rutas)
display(df_rutas)

### **Consulta 4: Análisis de propinas**

Exploraremos la relación entre el monto del viaje y la propina.

In [ ]:
query_propinas = """
SELECT 
    CASE 
        WHEN total_amount < 10 THEN '0-10'
        WHEN total_amount < 20 THEN '10-20'
        WHEN total_amount < 30 THEN '20-30'
        WHEN total_amount < 50 THEN '30-50'
        ELSE '50+'
    END AS rango_monto,
    COUNT(*) AS num_viajes,
    ROUND(AVG(tip_amount), 2) AS propina_promedio,
    ROUND(AVG(tip_amount / total_amount) * 100, 2) AS porcentaje_propina_promedio,
    ROUND(SUM(tip_amount), 2) AS propina_total
FROM taxi_viajes
WHERE total_amount > 0 AND tip_amount >= 0
GROUP BY rango_monto
ORDER BY 
    CASE rango_monto
        WHEN '0-10' THEN 1
        WHEN '10-20' THEN 2
        WHEN '20-30' THEN 3
        WHEN '30-50' THEN 4
        ELSE 5
    END
"""

df_propinas = spark.sql(query_propinas)
display(df_propinas)

---
## **Parte 3: Visualizaciones Rápidas en el Notebook**

Databricks tiene capacidades de visualización integradas. Podemos generar gráficos directamente desde los DataFrames usando `display()`.

### **Visualización 1: Viajes por día de la semana (gráfico de barras)**

Usa el resultado de `df_dia_semana` y haz clic en el icono de gráfico de barras en la salida de `display()`. Configura:
*   **Clave**: dia_semana
*   **Valores**: num_viajes
*   **Tipo de gráfico**: Barras

También podemos crear visualizaciones con matplotlib para mayor control.

In [ ]:
# Convertir a pandas para visualización con matplotlib (solo para muestras pequeñas)
pdf_dia_semana = df_dia_semana.toPandas()

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.barplot(data=pdf_dia_semana, x='dia_semana', y='num_viajes', palette='viridis')
plt.title('Número de Viajes por Día de la Semana')
plt.xlabel('Día de la Semana')
plt.ylabel('Número de Viajes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### **Visualización 2: Distribución de viajes por hora (línea)**

In [ ]:
pdf_hora = df_hora.toPandas()

plt.figure(figsize=(12, 5))
plt.plot(pdf_hora['hora'], pdf_hora['num_viajes'], marker='o', linestyle='-', color='coral', linewidth=2)
plt.title('Número de Viajes por Hora del Día')
plt.xlabel('Hora')
plt.ylabel('Número de Viajes')
plt.grid(True, alpha=0.3)
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

### **Visualización 3: Relación entre distancia y monto (scatter plot)**

Para evitar sobrecargar el gráfico, tomamos una muestra.

In [ ]:
# Tomar una muestra aleatoria del 1% para visualización
df_sample = df_taxi.sample(fraction=0.01, seed=42).select("trip_distance", "total_amount", "tip_amount")
pdf_sample = df_sample.toPandas()

plt.figure(figsize=(10, 6))
plt.scatter(pdf_sample['trip_distance'], pdf_sample['total_amount'], 
            alpha=0.3, c=pdf_sample['tip_amount'], cmap='viridis', s=10)
plt.colorbar(label='Propina ($)')
plt.title('Relación entre Distancia y Monto Total del Viaje')
plt.xlabel('Distancia (millas)')
plt.ylabel('Monto Total ($)')
plt.xlim(0, 30)  # Limitar para mejor visualización
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## **Ejercicios Prácticos para el Estudiante**

Ahora aplica lo aprendido con estos ejercicios basados en el dataset de NYC Taxi.

### **Ejercicio 1: Análisis de pagos con tarjeta vs efectivo**

El dataset tiene una columna `payment_type` con los siguientes códigos:
*   1 = Tarjeta de crédito
*   2 = Efectivo
*   3 = Sin cargo
*   4 = Disputa
*   5 = Desconocido

Crea una consulta SQL que calcule para cada tipo de pago:
*   Número de viajes
*   Porcentaje del total de viajes
*   Monto promedio
*   Propina promedio (para tarjeta vs efectivo, ¿hay diferencia?)

In [ ]:
# --- INICIO DE TU CÓDIGO ---

query_pago = """
SELECT 
    payment_type,
    CASE payment_type
        WHEN 1 THEN 'Tarjeta'
        WHEN 2 THEN 'Efectivo'
        WHEN 3 THEN 'Sin cargo'
        WHEN 4 THEN 'Disputa'
        ELSE 'Desconocido'
    END AS tipo_pago,
    COUNT(*) AS num_viajes,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM taxi_viajes), 2) AS porcentaje,
    ROUND(AVG(total_amount), 2) AS monto_promedio,
    ROUND(AVG(tip_amount), 2) AS propina_promedio
FROM taxi_viajes
GROUP BY payment_type
ORDER BY payment_type
"""

df_pago = spark.sql(query_pago)
display(df_pago)

# Visualización
pdf_pago = df_pago.toPandas()
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.pie(pdf_pago['num_viajes'], labels=pdf_pago['tipo_pago'], autopct='%1.1f%%')
plt.title('Distribución por Tipo de Pago')

plt.subplot(1, 2, 2)
sns.barplot(data=pdf_pago, x='tipo_pago', y='propina_promedio')
plt.title('Propina Promedio por Tipo de Pago')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 2: Viajes por aeropuerto**

Las zonas de recogida (PULocationID) incluyen los aeropuertos JFK y LaGuardia. Investiga los códigos de zona (puedes buscar en internet o usar la tabla de zonas disponible en Databricks). Luego, calcula:
*   Número de viajes desde/hacia JFK
*   Número de viajes desde/hacia LaGuardia
*   Distancia promedio de estos viajes
*   Compara con viajes que no son de aeropuerto

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# Códigos de zona aproximados (pueden variar según el año)
# JFK: 132, LaGuardia: 138 (verificar con datos actuales)

query_aeropuertos = """
SELECT 
    CASE 
        WHEN PULocationID IN (132, 138) OR DOLocationID IN (132, 138) THEN 'Aeropuerto'
        ELSE 'Otros'
    END AS tipo_viaje,
    COUNT(*) AS num_viajes,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio,
    ROUND(AVG(total_amount), 2) AS monto_promedio
FROM taxi_viajes
GROUP BY tipo_viaje
"""

df_aeropuertos = spark.sql(query_aeropuertos)
display(df_aeropuertos)

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 3: Ventana de tiempo - viajes por mes**

Extrae el mes de `tpep_pickup_datetime` y calcula:
*   Total de viajes por mes
*   Ingresos totales por mes
*   Variación porcentual respecto al mes anterior

*Pista: Puedes usar funciones de ventana (LAG) para calcular la variación.*

In [ ]:
# --- INICIO DE TU CÓDIGO ---

from pyspark.sql.functions import month, year, sum, lag
from pyspark.sql.window import Window

# Crear columna de mes y año
df_taxi_mes = df_taxi.withColumn("anio", year("tpep_pickup_datetime")) \
                     .withColumn("mes", month("tpep_pickup_datetime"))

# Agrupar por mes
df_mes = df_taxi_mes.groupBy("anio", "mes").agg(
    count("*").alias("num_viajes"),
    sum("total_amount").alias("ingresos_totales")
).orderBy("anio", "mes")

# Calcular variación
windowSpec = Window.orderBy("anio", "mes")
df_mes_var = df_mes.withColumn(
    "viajes_mes_anterior", 
    lag("num_viajes").over(windowSpec)
).withColumn(
    "variacion_porcentual",
    (col("num_viajes") - col("viajes_mes_anterior")) / col("viajes_mes_anterior") * 100
)

display(df_mes_var)

# Visualización
pdf_mes = df_mes_var.toPandas()
pdf_mes['anio_mes'] = pdf_mes['anio'].astype(str) + '-' + pdf_mes['mes'].astype(str).str.zfill(2)

plt.figure(figsize=(12, 5))
plt.bar(pdf_mes['anio_mes'], pdf_mes['num_viajes'], color='skyblue')
plt.title('Número de Viajes por Mes')
plt.xlabel('Mes')
plt.ylabel('Número de Viajes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- FIN DE TU CÓDIGO ---

---
## **Conclusiones**

En esta sesión práctica hemos:
1. **Cargado** un dataset real de gran tamaño (NYC Taxi) desde los datasets de muestra de Databricks.
2. **Explorado** su estructura y realizado consultas SQL complejas con PySpark.
3. **Creado** visualizaciones dentro del notebook para analizar patrones de viajes.
4. **Descubierto** insights como:
   * Los viernes son los días con más viajes.
   * Las horas pico son por la mañana (8-9 AM) y por la tarde (5-7 PM).
   * Existe una fuerte correlación entre distancia y monto del viaje.
   * Los pagos con tarjeta reciben propinas significativamente mayores que en efectivo.

**Conceptos clave reforzados:**
*   **PySpark SQL** permite analizar grandes volúmenes de datos de manera declarativa.
*   Las **visualizaciones** son fundamentales para comunicar hallazgos.
*   Los **datasets reales** contienen imperfecciones y requieren limpieza (valores nulos, outliers).

¡Has completado tu primer análisis de Big Data en Databricks!